In [44]:
import optuna
import numpy as np
import pandas as pd
import psycopg as pg
import mlflow
import os
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import matplotlib.pyplot as plt

In [29]:
TABLE_NAME = "clean_users_churn"
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = "SFS_analyzing"
RUN_NAME = "model_bayesian_search"
REGISTRY_MODEL_NAME = "SFS_MODEL"

In [3]:
configs = {"sslmode": "require", "target_session_attrs": "read-write"}
db_credits = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD")
}

configs.update(db_credits)

In [5]:
with pg.connect(**configs) as conn:
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")

        data = cur.fetchall()

        columns = [col.name for col in cur.description]

df = pd.DataFrame(data=data, columns=columns)

In [6]:
df.head(3)

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,28,1680-VDCWW,2019-02-01,NaT,One year,No,Bank transfer (automatic),19.80,202.25,Fiber optic,...,No,No,No,No,Male,0,Yes,No,No,0
1,29,1066-JKSGK,2019-11-01,2019-12-01,Month-to-month,No,Mailed check,20.15,20.15,Fiber optic,...,No,No,No,No,Male,0,No,No,No,1
2,30,3638-WEABW,2015-04-01,NaT,Two year,Yes,Credit card (automatic),59.90,3505.10,DSL,...,No,Yes,No,No,Female,0,Yes,No,Yes,0


In [7]:
X = df.drop(columns=["id", "customer_id", "begin_date", "end_date", "target"])
y = df["target"]

In [18]:
num_columns = ["monthly_charges", "total_charges", "senior_citizen"]
cat_columns = X.drop(columns=num_columns).columns.tolist()

In [25]:
encoder_oh = OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False)
scaler = StandardScaler()

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [26]:
preprocess = ColumnTransformer(
    transformers=[
        ("cat", encoder_oh, cat_columns),
        ("num", scaler, num_columns)
    ]
)

In [28]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

In [30]:
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

In [31]:
storage = "sqlite:///local.study.db"
study_name = "churn_model"

In [33]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [47]:
from sklearn.base import defaultdict
from numpy import median, array
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, f1_score, recall_score, log_loss


def objective(trial: optuna.Trial) -> float:
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
        "depth": trial.suggest_int("depth", 1, 12),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.1, 5),
        "random_strength": trial.suggest_float("random_strength", 0.1, 5),
        "loss_function": "Logloss",
        "task_type": "CPU",
        "random_seed": 0,
        "iterations": 300,
        "verbose": False
    }

    model = CatBoostClassifier(**params)
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocess),
        ("model", model)
    ])
    skf = StratifiedKFold(n_splits=2)
    metrics = defaultdict(list)

    for i, (train_index, val_index) in enumerate(skf.split(X_train, y_train)):
        train_x = X_train.iloc[train_index]
        train_y = y_train.iloc[train_index]
        val_x = X_train.iloc[val_index]
        val_y = y_train.iloc[val_index]

        pipeline.fit(train_x, train_y)

        prediction = pipeline.predict(val_x)
        probas = pipeline.predict_proba(val_x)[:, 1]

        _, err1, _, err2 = confusion_matrix(val_y, prediction, normalize="all").ravel()
        auc = roc_auc_score(val_y, probas)
        precision = precision_score(val_y, prediction)
        recall = recall_score(val_y, prediction)
        f1 = f1_score(val_y, prediction)
        logloss = log_loss(val_y, prediction)

        metrics["err1"].append(err1)
        metrics["err2"].append(err2)
        metrics["auc"].append(auc)
        metrics["precision"].append(precision)
        metrics["recall"].append(recall)
        metrics["f1"].append(f1)
        metrics["logloss"].append(logloss)


    auc = median(array(metrics["auc"]))

    return auc

In [37]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if not experiment:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

In [ ]:
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id


In [68]:
from optuna.integration.mlflow import MLflowCallback

mlflc = MLflowCallback(tracking_uri=f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}",
                       metric_name="AUC",
                       create_experiment=False,
                       mlflow_kwargs={"experiment_id": experiment_id,
                                      "tags": {"mlflow.parentRunId": run_id}}
                       )

/tmp/ipykernel_1680/2022215359.py:3: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflc = MLflowCallback(tracking_uri=f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}",


In [69]:
study = optuna.create_study(direction="maximize",
                            study_name=study_name,
                            storage=storage,
                            sampler=optuna.samplers.TPESampler(),
                            load_if_exists=True)
study.optimize(objective, n_trials=10, callbacks=[mlflc])
best_params = study.best_params

print(f"Number of finished trials: ${len(study.trials)}")
print(f"Best params: {best_params}")

Number of finished trials: $43
Best params: {'learning_rate': 0.0361704350818663, 'depth': 4, 'l2_leaf_reg': 0.3769325461232931, 'random_strength': 4.119256243043063}


In [54]:
run_id

'72a6ce433f30427e8f4553878ccf679c'

In [55]:
experiment_id

'7'

In [63]:
model = CatBoostClassifier(**best_params)
pipeline = Pipeline(steps=[
    ("cat", preprocess),
    ("model", model)
])
pipeline.fit(X_train, y_train)

0:	learn: 0.6726308	total: 1.98ms	remaining: 1.98s
1:	learn: 0.6611258	total: 3.49ms	remaining: 1.74s
2:	learn: 0.6487617	total: 4.95ms	remaining: 1.65s
3:	learn: 0.6335210	total: 6.47ms	remaining: 1.61s
4:	learn: 0.6210524	total: 8.25ms	remaining: 1.64s
5:	learn: 0.6098133	total: 9.76ms	remaining: 1.62s
6:	learn: 0.6002504	total: 11.1ms	remaining: 1.58s
7:	learn: 0.5881663	total: 13ms	remaining: 1.62s
8:	learn: 0.5755828	total: 14.9ms	remaining: 1.64s
9:	learn: 0.5663227	total: 16.8ms	remaining: 1.66s
10:	learn: 0.5564797	total: 18.9ms	remaining: 1.7s
11:	learn: 0.5495519	total: 20.4ms	remaining: 1.68s
12:	learn: 0.5405759	total: 22.2ms	remaining: 1.69s
13:	learn: 0.5328473	total: 23.8ms	remaining: 1.68s
14:	learn: 0.5247745	total: 25.7ms	remaining: 1.69s
15:	learn: 0.5179676	total: 27.5ms	remaining: 1.69s
16:	learn: 0.5121152	total: 29.5ms	remaining: 1.7s
17:	learn: 0.5087555	total: 30.9ms	remaining: 1.69s
18:	learn: 0.5038129	total: 32.5ms	remaining: 1.68s
19:	learn: 0.4977598	total

Pipeline(steps=[('cat',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(drop='if_binary',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['type', 'paperless_billing',
                                                   'payment_method',
                                                   'internet_service',
                                                   'online_security',
                                                   'online_backup',
                                                   'device_protection',
                                                   'tech_support',
                                                   'streaming_tv',
                                                   'streaming_movies', 'gender',
                                                   'partner', 'dependents',
                                                   'multiple_lines']),
                                                 ('num', StandardScaler(),
                                                  ['monthly_charges',
                                                   'total_charges',
                                                   'senior_citizen'])])),
                ('model',
                 <catboost.core.CatBoostClassifier object at 0x7f58be01d150>)])

In [66]:
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.log_metrics(best_params)
    cv_info = mlflow.sklearn.log_model(sk_model=pipeline, artifact_path="cv")

In [67]:
run_id

'527c830fa10d40c4b24c5459731bdd68'